In [ ]:
# ──────────────────────────────────────────────────────────────────
# CELL 1: Mount Google Drive
# ──────────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted.")

Mounted at /content/drive
Drive mounted.


In [ ]:
# ──────────────────────────────────────────────────────────────────
# CELL 2: Configure paths + verify data
# ──────────────────────────────────────────────────────────────────
import os

# ⚠️ EDIT THESE PATHS to match your Google Drive folder structure
ORIGINAL_PDB_DIR = '/content/drive/MyDrive/DATA-RECON-PROTOCOL/STANFORD-RNA/try1-small/PDB_Files'
WORK_DIR         = '/content/stanford_rna_WORK'
PROCESSED_DIR    = os.path.join(WORK_DIR, 'processed_output')

os.makedirs(PROCESSED_DIR, exist_ok=True)

if os.path.exists(ORIGINAL_PDB_DIR):
    all_cif_files = sorted(
        [f for f in os.listdir(ORIGINAL_PDB_DIR) if f.endswith('.cif')]
    )
    total_files = len(all_cif_files)
    print(f"Found {total_files:,} CIF files in {ORIGINAL_PDB_DIR}")
    print(f"First 5: {all_cif_files[:5]}")
    print(f"Last 5:  {all_cif_files[-5:]}")
else:
    print(f"ERROR: Folder not found: {ORIGINAL_PDB_DIR}")
    print("Scanning your Drive for .cif files...")
    for item in sorted(os.listdir('/content/drive/MyDrive')):
        full = os.path.join('/content/drive/MyDrive', item)
        if os.path.isdir(full):
            try:
                cifs = [f for f in os.listdir(full) if f.endswith('.cif')]
                if cifs:
                    print(f"  {item}/ -> {len(cifs)} .cif files found")
            except:
                pass
    print("Update ORIGINAL_PDB_DIR above and re-run this cell.")



Found 130 CIF files in /content/drive/MyDrive/DATA-RECON-PROTOCOL/STANFORD-RNA/try1-small/PDB_Files
First 5: ['1a1t.cif', '1a34.cif', '1a3m.cif', '1a4d.cif', '1a4t.cif']
Last 5:  ['1euy.cif', '1evp.cif', '1evv.cif', '1exd.cif', '1exy.cif']


In [ ]:
# ──────────────────────────────────────────────────────────────────
# CELL 3: Install BioPython
# ──────────────────────────────────────────────────────────────────
!pip install biopython --quiet
print("BioPython installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 33.2 MB/s eta 0:00:00
BioPython installed.


In [ ]:
# ──────────────────────────────────────────────────────────────────
# CELL 4: Quick test on 10 files (validate before full run)
# ──────────────────────────────────────────────────────────────────
import numpy as np
import time
from Bio.PDB.MMCIFParser import MMCIFParser

cif_parser = MMCIFParser(QUIET=True)


def extract_rna_data(cif_path):
    """
    Extract C1' atom coordinates and RNA sequence from one CIF file.

    Walks the structure hierarchy: Structure > Model > Chain > Residue > Atom
    For each RNA nucleotide (A, U, G, C), finds the C1' atom and records
    its XYZ coordinates.

    Returns:
        dict: {chain_id: {'coords': np.array(N,3), 'sequence': str, 'length': int}}
        Only chains with >= 10 nucleotides are returned.
    """
    structure = cif_parser.get_structure('rna', cif_path)
    chains_data = {}

    for model in structure:
        for chain in model:
            chain_id = chain.get_id()
            coords = []
            seq_letters = []

            for residue in chain:
                resname = residue.get_resname().strip()
                if resname in ['A', 'U', 'G', 'C']:
                    for atom in residue:
                        if atom.get_name() == "C1'":
                            coords.append(atom.get_vector().get_array())
                            seq_letters.append(resname)
                            break

            if len(coords) >= 10:
                chains_data[chain_id] = {
                    'coords': np.array(coords, dtype=np.float32),
                    'sequence': ''.join(seq_letters),
                    'length': len(coords)
                }
        break  # first model only
    return chains_data


# Quick test
print("Testing on first 10 files...")
test_count = 0
start = time.time()

for filename in all_cif_files[:10]:
    filepath = os.path.join(ORIGINAL_PDB_DIR, filename)
    try:
        chains = extract_rna_data(filepath)
        for cid, data in chains.items():
            test_count += 1
            if test_count <= 2:
                print(f"  {filename} chain {cid}: "
                      f"seq={data['sequence'][:40]}... "
                      f"len={data['length']} "
                      f"coords={data['coords'].shape}")
    except Exception as e:
        print(f"  Error (expected for protein-only files): {filename}: {e}")

elapsed = time.time() - start
rate = 10 / elapsed if elapsed > 0 else 0
print(f"\nTest passed: {test_count} templates from 10 files in {elapsed:.1f}s")
print(f"Speed: {rate:.1f} files/sec")
print(f"Estimated full run ({total_files:,} files): {total_files/rate/3600:.1f} hours")



Testing on first 10 files...
  1a1t.cif chain B: seq=GGACUAGCGGAGGCUAGUCC... len=20 coords=(20, 3)
  1a34.cif chain B: seq=AAAAAAAAAA... len=10 coords=(10, 3)

Test passed: 14 templates from 10 files in 13.1s
Speed: 0.8 files/sec
Estimated full run (130 files): 0.0 hours


In [ ]:
# ──────────────────────────────────────────────────────────────────
# CELL 5: FULL EXTRACTION WITH CHECKPOINTING
# ──────────────────────────────────────────────────────────────────
# Processes ALL CIF files. If Colab disconnects:
#   1. Reconnect
#   2. Re-run Cell 1 (mount), Cell 2 (paths), Cell 3 (biopython)
#   3. Re-run this cell -> it resumes from checkpoint automatically
# ──────────────────────────────────────────────────────────────────
import pickle
import numpy as np
import time
import os
from Bio.PDB.MMCIFParser import MMCIFParser

cif_parser = MMCIFParser(QUIET=True)


def extract_rna_data(cif_path):
    """Extract C1' coordinates and sequence from a CIF file."""
    structure = cif_parser.get_structure('rna', cif_path)
    chains_data = {}
    for model in structure:
        for chain in model:
            chain_id = chain.get_id()
            coords = []
            seq_letters = []
            for residue in chain:
                resname = residue.get_resname().strip()
                if resname in ['A', 'U', 'G', 'C']:
                    for atom in residue:
                        if atom.get_name() == "C1'":
                            coords.append(atom.get_vector().get_array())
                            seq_letters.append(resname)
                            break
            if len(coords) >= 10:
                chains_data[chain_id] = {
                    'coords': np.array(coords, dtype=np.float32),
                    'sequence': ''.join(seq_letters),
                    'length': len(coords)
                }
        break
    return chains_data


# Paths

ORIGINAL_PDB_DIR = '/content/drive/MyDrive/DATA-RECON-PROTOCOL/STANFORD-RNA/try1-small/PDB_Files'
WORK_DIR         = '/content/stanford_rna_WORK'
PROCESSED_DIR    = os.path.join(WORK_DIR, 'processed_output')
os.makedirs(PROCESSED_DIR, exist_ok=True)

CHECKPOINT_PATH = os.path.join(PROCESSED_DIR, 'full_checkpoint.pkl')
FINAL_PKL_PATH  = os.path.join(PROCESSED_DIR, 'rna_templates.pkl')
FINAL_FASTA     = os.path.join(PROCESSED_DIR, 'rna_templates.fasta')

# Scan files
all_cif_files = sorted(
    [f for f in os.listdir(ORIGINAL_PDB_DIR) if f.endswith('.cif')]
)
total_files = len(all_cif_files)

# Load checkpoint if resuming
if os.path.exists(CHECKPOINT_PATH):
    with open(CHECKPOINT_PATH, 'rb') as f:
        ckpt = pickle.load(f)
    all_templates = ckpt['templates']
    processed_files = ckpt['processed_files']
    print(f"RESUMED: {len(processed_files):,} done, "
          f"{len(all_templates):,} templates")
else:
    all_templates = {}
    processed_files = set()
    print("Starting fresh")

remaining = [f for f in all_cif_files if f not in processed_files]
print(f"Total: {total_files:,} | Done: {len(processed_files):,} | "
      f"Remaining: {len(remaining):,}")

errors = []
start_time = time.time()
SAVE_EVERY = 500

for i, filename in enumerate(remaining):
    filepath = os.path.join(ORIGINAL_PDB_DIR, filename)
    pdb_id = filename.split('.')[0].upper()

    try:
        chains = extract_rna_data(filepath)
        for chain_id, data in chains.items():
            template_id = f"{pdb_id}_{chain_id}"
            all_templates[template_id] = data
    except Exception as e:
        errors.append((filename, str(e)))

    processed_files.add(filename)

    if (i + 1) % SAVE_EVERY == 0:
        with open(CHECKPOINT_PATH, 'wb') as f:
            pickle.dump({
                'templates': all_templates,
                'processed_files': processed_files
            }, f, protocol=4)
        elapsed = time.time() - start_time
        rate = (i + 1) / elapsed
        eta_hrs = (len(remaining) - i - 1) / rate / 3600
        print(f"  [{len(processed_files):,}/{total_files:,}] "
              f"Templates: {len(all_templates):,} | "
              f"{rate:.1f} files/sec | ETA: {eta_hrs:.1f}h | "
              f"Errors: {len(errors)} | checkpoint saved")

# Final save: pickle
with open(FINAL_PKL_PATH, 'wb') as f:
    pickle.dump(all_templates, f, protocol=4)

# Final save: FASTA (needed for MMseqs2 in Phase 2)
with open(FINAL_FASTA, 'w') as f:
    for tid, data in all_templates.items():
        f.write(f">{tid}\n{data['sequence']}\n")

pkl_mb = os.path.getsize(FINAL_PKL_PATH) / 1e6
print(f"\nPHASE 1 COMPLETE")
print(f"  Templates: {len(all_templates):,}")
print(f"  Pickle: {FINAL_PKL_PATH} ({pkl_mb:.1f} MB)")
print(f"  FASTA:  {FINAL_FASTA}")
print(f"  Errors: {len(errors)}")

if os.path.exists(CHECKPOINT_PATH):
    os.remove(CHECKPOINT_PATH)

if errors:
    err_path = os.path.join(PROCESSED_DIR, 'errors.txt')
    with open(err_path, 'w') as f:
        for fname, err in errors:
            f.write(f"{fname}: {err}\n")
    print(f"  Error log: {err_path}")


# ──────────────────────────────────────────────────────────────────
# CELL 6: Verify Phase 1 output
# ──────────────────────────────────────────────────────────────────
import pickle
import os

PROCESSED_DIR = '/content/stanford_rna_WORK/processed_output'
pkl_path = os.path.join(PROCESSED_DIR, 'rna_templates.pkl')

with open(pkl_path, 'rb') as f:
    templates = pickle.load(f)

lengths = [t['length'] for t in templates.values()]
print(f"Templates: {len(templates):,} ({os.path.getsize(pkl_path)/1e6:.1f} MB)")
print(f"Length range: {min(lengths)}–{max(lengths)} nt")
print(f"Mean length: {sum(lengths)/len(lengths):.1f} nt")

for tid, data in list(templates.items())[:3]:
    print(f"  {tid}: len={data['length']} seq={data['sequence'][:40]}...")



Starting fresh
Total: 130 | Done: 0 | Remaining: 130

PHASE 1 COMPLETE
  Templates: 140
  Pickle: /content/stanford_rna_WORK/processed_output/rna_templates.pkl (0.1 MB)
  FASTA:  /content/stanford_rna_WORK/processed_output/rna_templates.fasta
  Errors: 0
Templates: 140 (0.1 MB)
Length range: 10–2891 nt
Mean length: 49.0 nt
  1A1T_B: len=20 seq=GGACUAGCGGAGGCUAGUCC...
  1A34_B: len=10 seq=AAAAAAAAAA...
  1A34_C: len=10 seq=UUUUUUUUUU...


In [ ]:
# ═══════════════════════════════════════════════════════════════════
#                    PHASE 2: MMSEQS2 TEMPLATE SEARCH
#       (Input: FASTA + test_sequences.csv → Output: results.m8)
# ═══════════════════════════════════════════════════════════════════


# ──────────────────────────────────────────────────────────────────
# CELL 7: Install MMseqs2
# ──────────────────────────────────────────────────────────────────
!apt-get update -qq && apt-get install -y -qq mmseqs2 2>/dev/null
!mmseqs version
print("MMseqs2 installed.")


# ──────────────────────────────────────────────────────────────────
# CELL 8: Prepare test sequences as FASTA
# ──────────────────────────────────────────────────────────────────
import pandas as pd
import os

PROCESSED_DIR = '/content/stanford_rna_WORK/processed_output'
FASTA_PATH    = os.path.join(PROCESSED_DIR, 'rna_templates.fasta')
MMSEQS_DIR    = '/content/mmseqs_work_v1'
os.makedirs(MMSEQS_DIR, exist_ok=True)

# ⚠️ EDIT THIS: path to competition's test_sequences.csv
# For Colab: upload to Drive first, or upload directly via Colab file upload
# For Kaggle: /kaggle/input/stanford-rna-3d-folding-2/test_sequences.csv
TEST_CSV = '/content/stanford_rna_WORK/processed_output/test_sequences .csv'

test_df = pd.read_csv(TEST_CSV)
print(f"test_sequences.csv loaded: {test_df.shape}")
print(f"Columns: {list(test_df.columns)}")
print(test_df.head(3))

# ── Determine the correct column names by inspecting the CSV ──
# The competition CSV has target_id and sequence columns.
# Detect them automatically:
target_col = None
seq_col = None
for col in test_df.columns:
    col_lower = col.lower().strip()
    if 'target' in col_lower and 'id' in col_lower:
        target_col = col
    elif col_lower == 'sequence':
        seq_col = col

if target_col is None or seq_col is None:
    print(f"\nWARNING: Could not auto-detect columns.")
    print(f"Available columns: {list(test_df.columns)}")
    print(f"Please set target_col and seq_col manually below.")
    # Fallback: use first two columns
    target_col = test_df.columns[0]
    seq_col = test_df.columns[1]

print(f"\nUsing: target_col='{target_col}', seq_col='{seq_col}'")

# ── Build lookup: target_id -> sequence ──
test_sequences = {}
for target_id, group in test_df.groupby(target_col):
    seq = group.iloc[0][seq_col]
    if isinstance(seq, str) and len(seq) > 0:
        test_sequences[target_id] = seq

print(f"Unique test targets: {len(test_sequences)}")

# ── Write test FASTA ──
test_fasta = os.path.join(MMSEQS_DIR, 'test_sequences.fasta')
with open(test_fasta, 'w') as f:
    for tid, seq in test_sequences.items():
        f.write(f">{tid}\n{seq}\n")

print(f"Test FASTA written: {test_fasta} ({len(test_sequences)} sequences)")


# ──────────────────────────────────────────────────────────────────
# CELL 9: Build MMseqs2 database + run search
# ──────────────────────────────────────────────────────────────────
import os

MMSEQS_DIR    = '/content/mmseqs_work_v1'
PROCESSED_DIR = '/content/stanford_rna_WORK/processed_output'
FASTA_PATH    = os.path.join(PROCESSED_DIR, 'rna_templates.fasta')

# Copy FASTA to local disk (faster than reading from Drive)
local_fasta = os.path.join(MMSEQS_DIR, 'pdb_rna.fasta')
if not os.path.exists(local_fasta):
    !cp "{FASTA_PATH}" "{local_fasta}"
    print(f"FASTA copied to local disk: {local_fasta}")

pdb_db     = os.path.join(MMSEQS_DIR, 'pdb_rna_db')
test_db    = os.path.join(MMSEQS_DIR, 'test_db')
test_fasta = os.path.join(MMSEQS_DIR, 'test_sequences.fasta')
results_db = os.path.join(MMSEQS_DIR, 'results')
results_m8 = os.path.join(MMSEQS_DIR, 'results.m8')
tmp_dir    = os.path.join(MMSEQS_DIR, 'tmp')
os.makedirs(tmp_dir, exist_ok=True)

# Step 1: Create databases
print("=" * 60)
print("STEP 1: Creating MMseqs2 databases...")
!mmseqs createdb "{local_fasta}" "{pdb_db}"
print("PDB template database created.")

!mmseqs createdb "{test_fasta}" "{test_db}"
print("Test sequence database created.")

# STEP 2: Create index  ← ADD --search-type 3
!mmseqs createindex "{pdb_db}" "{tmp_dir}" --search-type 3


# Step 3: Search
# Parameters (from winning Part 1 approaches):
#   -s 7.5          = high sensitivity (finds distant homologs)
#   --search-type 3 = nucleotide mode
#   -e 10.0         = permissive E-value (keep weak hits)
#   --max-seqs 50   = return top 50 hits per query
print("\nSTEP 3: Searching test sequences against PDB templates...")
!mmseqs search \
    "{test_db}" "{pdb_db}" "{results_db}" "{tmp_dir}" \
    -s 7.5 \
    --search-type 3 \
    -e 10.0 \
    --max-seqs 50
print("Search complete.")

# STEP 4: Convert to .m8  ← ADD --search-type 3
!mmseqs convertalis "{test_db}" "{pdb_db}" "{results_db}" "{results_m8}" \
    --search-type 3 \
    --format-output "query,target,pident,alnlen,mismatch,gapopen,qstart,qend,tstart,tend,evalue,bits,qaln,taln"
print(".m8 file created.")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
13-45111+ds-2
MMseqs2 installed.
test_sequences.csv loaded: (28, 8)
Columns: ['target_id', 'sequence', 'temporal_cutoff', 'description', 'stoichiometry', 'all_sequences', 'ligand_ids', 'ligand_SMILES']
  target_id                                           sequence  \
0      8ZNQ                     ACCGUGACGGGCCUUUUGGCUAUACGCGGU   
1      9IWF  GGUGUAUAAGCUCAUUAAUACGGUUUGAGCGUUUCGACCAGGCAAC...   
2      9JGM  GGAAGGGGAGUAACUUCAUUGCCGGUCGAUCGUCAUUACGAUGUGU...   

  temporal_cutoff                                        description  \
0      2025-06-04  Solution structure of the complex of naphthyri...   
1      2025-06-04  crystal structure of P. beijingensis xanthine-...   
2      2025-06-04  The Escherichia coli yybp riboswitch as a tand...   

  stoichiometry                                      al

In [ ]:
# ──────────────────────────────────────────────────────────────────
# CELL 10: Parse and rank template hits
# ──────────────────────────────────────────────────────────────────
import pandas as pd

results_m8 = '/content/mmseqs_work_v1/results.m8'


columns = [
    'query', 'target', 'pident', 'alnlen', 'mismatch', 'gapopen',
    'qstart', 'qend', 'tstart', 'tend', 'evalue', 'bits', 'qaln', 'taln'
]
hits = pd.read_csv(results_m8, sep='\t', names=columns)

print(f"Total hits: {len(hits):,}")
print(f"Test sequences with hits: {hits['query'].nunique()}")
print(f"Templates matched: {hits['target'].nunique()}")

# Rank: highest sequence identity first, then longest alignment
hits = hits.sort_values(
    ['query', 'pident', 'alnlen', 'evalue'],
    ascending=[True, False, False, True]
)

# Top 5 templates per test sequence (we need 5 predictions)
top5 = hits.groupby('query').head(5)
print(f"Top 5 per query: {len(top5):,} rows")

# How many test sequences have NO hits?
matched_ids = set(hits['query'].unique())
no_hits = set(test_sequences.keys()) - matched_ids
print(f"\nWith hits:    {len(matched_ids)}")
print(f"Without hits: {len(no_hits)} (will use dummy coordinates)")

# Show example
if len(hits) > 0:
    ex = hits['query'].iloc[0]
    print(f"\nExample hits for {ex}:")
    print(hits[hits['query'] == ex][
        ['target', 'pident', 'alnlen', 'evalue']
    ].head())

Total hits: 0
Test sequences with hits: 0
Templates matched: 0
Top 5 per query: 0 rows

With hits:    0
Without hits: 28 (will use dummy coordinates)


In [ ]:
# ──────────────────────────────────────────────────────────────────
# DIAGNOSTIC: Why did MMseqs2 return 0 hits?
# Paste the FULL output of this cell back to Claude
# ──────────────────────────────────────────────────────────────────
import os

PROCESSED_DIR = '/content/stanford_rna_WORK/processed_output'
MMSEQS_DIR    = '/content/mmseqs_work_v1'

print("=" * 60)
print("DIAGNOSTIC: Why 0 hits?")
print("=" * 60)

# ── 1. PDB template FASTA (what we search AGAINST) ──
print("\n[1] PDB TEMPLATE FASTA (rna_templates.fasta)")
fasta_path = os.path.join(PROCESSED_DIR, 'rna_templates.fasta')
if os.path.exists(fasta_path):
    fasta_size = os.path.getsize(fasta_path)
    seq_count = 0
    total_bases = 0
    lengths = []
    print(f"  File size: {fasta_size:,} bytes ({fasta_size/1e6:.2f} MB)")

    with open(fasta_path) as f:
        current_seq = ""
        for line in f:
            line = line.strip()
            if line.startswith('>'):
                if current_seq:
                    lengths.append(len(current_seq))
                    total_bases += len(current_seq)
                current_seq = ""
                seq_count += 1
            else:
                current_seq += line
        if current_seq:
            lengths.append(len(current_seq))
            total_bases += len(current_seq)

    print(f"  Sequences: {seq_count:,}")
    print(f"  Total bases: {total_bases:,}")

    if seq_count > 0:
        print(f"  Length range: {min(lengths)}–{max(lengths)} nt")
        print(f"  Mean length: {sum(lengths)/len(lengths):.0f} nt")

    # Show first 10 lines
    print(f"\n  First 10 lines:")
    with open(fasta_path) as f:
        for i, line in enumerate(f):
            if i >= 10:
                break
            print(f"    {line.rstrip()}")

    # Check for problems
    print(f"\n  Checking for issues...")
    with open(fasta_path) as f:
        first_seq = ""
        found_header = False
        for line in f:
            if line.startswith('>'):
                if found_header:
                    break
                found_header = True
            elif found_header:
                first_seq += line.strip()
        if first_seq:
            chars = set(first_seq)
            print(f"    Characters in first sequence: {chars}")
            non_rna = chars - set('AUGCNX')
            if non_rna:
                print(f"    WARNING: Non-RNA characters found: {non_rna}")
                if 'T' in non_rna:
                    print(f"    >>> Contains 'T' (DNA thymine) — may need T→U conversion <<<")
else:
    print(f"  FILE NOT FOUND: {fasta_path}")
    print(f"  Phase 1 was never completed!")

# ── 2. Test sequence FASTA (what we search WITH) ──
print(f"\n\n[2] TEST SEQUENCE FASTA (test_sequences.fasta)")
test_fasta = os.path.join(MMSEQS_DIR, 'test_sequences.fasta')
if os.path.exists(test_fasta):
    test_size = os.path.getsize(test_fasta)
    test_count = 0
    test_bases = 0
    test_lengths = []

    with open(test_fasta) as f:
        current_seq = ""
        for line in f:
            line = line.strip()
            if line.startswith('>'):
                if current_seq:
                    test_lengths.append(len(current_seq))
                    test_bases += len(current_seq)
                current_seq = ""
                test_count += 1
            else:
                current_seq += line
        if current_seq:
            test_lengths.append(len(current_seq))
            test_bases += len(current_seq)

    print(f"  File size: {test_size:,} bytes ({test_size/1e6:.2f} MB)")
    print(f"  Sequences: {test_count}")
    print(f"  Total bases: {test_bases:,}")

    if test_count > 0:
        print(f"  Length range: {min(test_lengths)}–{max(test_lengths)} nt")

    print(f"\n  First 10 lines:")
    with open(test_fasta) as f:
        for i, line in enumerate(f):
            if i >= 10:
                break
            print(f"    {line.rstrip()}")

    # Check characters
    with open(test_fasta) as f:
        first_seq = ""
        found_header = False
        for line in f:
            if line.startswith('>'):
                if found_header:
                    break
                found_header = True
            elif found_header:
                first_seq += line.strip()
        if first_seq:
            chars = set(first_seq)
            print(f"\n    Characters in first sequence: {chars}")
else:
    print(f"  FILE NOT FOUND: {test_fasta}")

# ── 3. Pickle check ──
print(f"\n\n[3] PICKLE FILE (rna_templates.pkl)")
pkl_path = os.path.join(PROCESSED_DIR, 'rna_templates.pkl')
if os.path.exists(pkl_path):
    import pickle
    pkl_size = os.path.getsize(pkl_path)
    print(f"  File size: {pkl_size:,} bytes ({pkl_size/1e6:.2f} MB)")
    with open(pkl_path, 'rb') as f:
        templates = pickle.load(f)
    print(f"  Templates: {len(templates):,}")
    if len(templates) > 0:
        for tid, data in list(templates.items())[:3]:
            print(f"    {tid}: len={data['length']} seq={data['sequence'][:50]}")
else:
    print(f"  NOT FOUND")

# ── 4. PDB_RNA folder check ──
print(f"\n\n[4] PDB_RNA FOLDER")
PDB_DIR = '/content/drive/MyDrive/DATA-RECON-PROTOCOL/STANFORD-RNA/try1-small/PDB_Files'
if os.path.exists(PDB_DIR):
    all_items = os.listdir(PDB_DIR)
    ext_counts = {}
    for f in all_items:
        if os.path.isdir(os.path.join(PDB_DIR, f)):
            ext_counts['<DIR>'] = ext_counts.get('<DIR>', 0) + 1
        elif '.' in f:
            ext = f.rsplit('.', 1)[-1].lower()
            ext_counts[ext] = ext_counts.get(ext, 0) + 1
        else:
            ext_counts['<no_ext>'] = ext_counts.get('<no_ext>', 0) + 1
    print(f"  Total items: {len(all_items):,}")
    print(f"  By type: {ext_counts}")
    print(f"  First 10: {sorted(all_items)[:10]}")
else:
    print(f"  NOT FOUND: {PDB_DIR}")

# ── 5. test_sequences.csv check ──
print(f"\n\n[5] TEST_SEQUENCES.CSV")
import pandas as pd
for path in [
    '/content/stanford_rna_WORK/processed_output/test_sequences.csv']:

    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"  Found: {path}")
        print(f"  Shape: {df.shape}")
        print(f"  Columns: {list(df.columns)}")
        print(f"  First 2 rows:")
        print(df.head(2).to_string())
        break
else:
    print(f"  NOT FOUND")

# ── 6. MMseqs2 database check ──
print(f"\n\n[6] MMSEQS2 DATABASE FILES")
if os.path.exists(MMSEQS_DIR):
    for f in sorted(os.listdir(MMSEQS_DIR)):
        fp = os.path.join(MMSEQS_DIR, f)
        if os.path.isfile(fp):
            size = os.path.getsize(fp)
            print(f"  {f}: {size:,} bytes")

print(f"\n{'=' * 60}")
print("Copy ALL output above and paste to Claude for analysis.")
print("=" * 60)

DIAGNOSTIC: Why 0 hits?

[1] PDB TEMPLATE FASTA (rna_templates.fasta)
  File size: 8,118 bytes (0.01 MB)
  Sequences: 140
  Total bases: 6,854
  Length range: 10–2891 nt
  Mean length: 49 nt

  First 10 lines:
    >1A1T_B
    GGACUAGCGGAGGCUAGUCC
    >1A34_B
    AAAAAAAAAA
    >1A34_C
    UUUUUUUUUU
    >1A3M_A
    GGCGUCACACCUUC
    >1A3M_B
    GGGUGAAGUCGCC

  Checking for issues...
    Characters in first sequence: {'C', 'A', 'G', 'U'}


[2] TEST SEQUENCE FASTA (test_sequences.fasta)
  File size: 9,958 bytes (0.01 MB)
  Sequences: 28
  Total bases: 9,762
  Length range: 19–4640 nt

  First 10 lines:
    >8ZNQ
    ACCGUGACGGGCCUUUUGGCUAUACGCGGU
    >9CFN
    AAGUACCCUCCAAGCCCUACAGGUUGGAAGAGGGGGCUAUCAGUCCUGUAGGCAGACUC
    >9E74
    GGUCCGUCCGGGUCCGUGGUUGCGUCGUUCAGGCGCCCGGUCUCCGGGUGCGCGAGCGACUAGCCGAUGCCAGUCGCAGGCAAAGCGGUCCACGUAACCCCGGCAACGGGGGAUCACGGUGCGGCUUAGACGUAAGUCCUGCCUCGUGGCCAUCGUCAGAUACGACGGCGUACGGGAGAAGGCCGGUAGUGGCAGAGAAGCUGCGGAAGCCUCAGCAGGCGAUUGUGGGGUCGAAGACCAGGUCGGCCGGACGGA
 


PDB_RNA folder: 130 files ← THIS IS THE PROBLEM Templates extracted: 145 ← from only 130 files Test sequences: 28 ← need to match against THOUSANDS Hits: 0 ← 145 tiny templates can't cover 28 test RNAs



In [ ]:
# ═══════════════════════════════════════════════════════════════════
#               PHASE 3: TRANSFER COORDINATES
#     (Input: templates + hits → Output: predicted 3D structures)
# ═══════════════════════════════════════════════════════════════════


# ──────────────────────────────────────────────────────────────────
# CELL 11: Coordinate transfer function + generate predictions
# ──────────────────────────────────────────────────────────────────
import numpy as np
import pickle
import pandas as pd

# Load templates
pkl_path = '/content/stanford_rna_WORK/processed_output/rna_templates.pkl'
with open(pkl_path, 'rb') as f:
    templates = pickle.load(f)
print(f"Loaded {len(templates):,} templates")


def transfer_coordinates(query_seq, template_data, qaln, taln, qstart, tstart):
    """
    Transfer C1' coordinates from template to query using alignment.

    For aligned positions: copy the template's C1' XYZ.
    For gaps/unaligned: interpolate linearly between nearest assigned coords.

    Args:
        query_seq:     Full query sequence string (e.g. "AUGCUUAG...")
        template_data: Dict with 'coords' (np.array), 'sequence', 'length'
        qaln:          Aligned query (with gaps "-") from MMseqs2
        taln:          Aligned template (with gaps "-") from MMseqs2
        qstart:        1-indexed start position in query (from MMseqs2)
        tstart:        1-indexed start position in template (from MMseqs2)

    Returns:
        np.array of shape (len(query_seq), 3) — predicted C1' coordinates
    """
    query_len = len(query_seq)
    template_coords = template_data['coords']
    pred = np.full((query_len, 3), np.nan, dtype=np.float32)

    q_pos = qstart - 1  # 0-indexed
    t_pos = tstart - 1

    for q_char, t_char in zip(qaln, taln):
        if q_char != '-' and t_char != '-':
            if t_pos < len(template_coords) and q_pos < query_len:
                pred[q_pos] = template_coords[t_pos]
            q_pos += 1
            t_pos += 1
        elif q_char == '-':
            t_pos += 1
        elif t_char == '-':
            q_pos += 1

    # Interpolate NaN positions
    assigned = np.where(~np.isnan(pred[:, 0]))[0]

    if len(assigned) == 0:
        # No coords assigned — place along a straight line
        pred = np.zeros((query_len, 3), dtype=np.float32)
        pred[:, 0] = np.arange(query_len) * 5.9  # ~5.9 A between C1' atoms
        return pred

    for dim in range(3):
        pred[:, dim] = np.interp(
            np.arange(query_len),
            assigned,
            pred[assigned, dim]
        )

    return pred


def add_noise(coords, scale=0.5):
    """Add small Gaussian noise to coordinates for diversity."""
    return coords + np.random.normal(0, scale, coords.shape).astype(np.float32)


# ── Generate up to 5 predictions per test sequence ──
# Strategy: use top 5 template hits. If fewer than 5 hits, add noise
# to the best prediction to create diversity.

predictions = {}  # target_id -> list of 5 np.arrays, each shape (seq_len, 3)

for target_id, query_seq in test_sequences.items():
    target_hits = top5[top5['query'] == target_id]
    preds_for_target = []

    for _, row in target_hits.iterrows():
        tid = row['target']
        if tid not in templates:
            continue
        try:
            coords = transfer_coordinates(
                query_seq=query_seq,
                template_data=templates[tid],
                qaln=str(row['qaln']),
                taln=str(row['taln']),
                qstart=int(row['qstart']),
                tstart=int(row['tstart'])
            )
            preds_for_target.append(coords)
        except Exception:
            continue

        if len(preds_for_target) >= 5:
            break

    # If no templates found, use a straight line as fallback
    if len(preds_for_target) == 0:
        seq_len = len(query_seq)
        dummy = np.zeros((seq_len, 3), dtype=np.float32)
        dummy[:, 0] = np.arange(seq_len) * 5.9
        preds_for_target.append(dummy)

    # Pad to exactly 5 predictions using noise on best prediction
    while len(preds_for_target) < 5:
        preds_for_target.append(add_noise(preds_for_target[0], scale=0.5))

    predictions[target_id] = preds_for_target[:5]

print(f"Predictions generated for {len(predictions)} targets")
print(f"Each target has exactly 5 predictions.")



Loaded 140 templates
Predictions generated for 28 targets
Each target has exactly 5 predictions.


In [ ]:
# ═══════════════════════════════════════════════════════════════════
#               PHASE 4: GENERATE SUBMISSION.CSV
#
# VERIFIED FORMAT (from Kaggle competition page):
#   Columns: ID, resname, resid, x_1, y_1, z_1, ..., x_5, y_5, z_5
#   Example: R1107_1,G,1,-7.561,9.392,9.361,...,-7.301,9.023,8.932
#
#   ID format: {target_id}_{position} where position is 1-indexed
#   resname: nucleotide letter (A, U, G, C)
#   resid: residue number (1-indexed)
#   x_N, y_N, z_N: C1' coordinates for prediction N (N = 1 to 5)
# ═══════════════════════════════════════════════════════════════════


# ──────────────────────────────────────────────────────────────────
# CELL 12: Build submission.csv
# ──────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import os

# ── Option A: Build from sample_submission.csv (RECOMMENDED) ──
# If you have the competition's sample_submission.csv, use it to get
# the exact IDs, resnames, and resids. This guarantees correct format.

SAMPLE_SUB_PATH = '/content/stanford_rna_WORK/sample_submission.csv'
# SAMPLE_SUB_PATH = '/kaggle/input/stanford-rna-3d-folding-2/sample_submission.csv'

if os.path.exists(SAMPLE_SUB_PATH):
    print("Using sample_submission.csv as template (recommended).")
    sample_sub = pd.read_csv(SAMPLE_SUB_PATH)
    print(f"  Shape: {sample_sub.shape}")
    print(f"  Columns: {list(sample_sub.columns)}")

    # Fill in coordinates from our predictions
    coord_cols = [f'{c}_{n}' for n in range(1, 6) for c in ['x', 'y', 'z']]

    for idx, row in sample_sub.iterrows():
        row_id = str(row['ID'])

        # Parse target_id and position from ID column
        # ID format: R1107_1 means target=R1107, position=1
        parts = row_id.rsplit('_', 1)
        if len(parts) == 2:
            target_id = parts[0]
            pos = int(parts[1]) - 1  # convert to 0-indexed
        else:
            continue

        if target_id in predictions:
            for n in range(5):  # 5 predictions
                coords = predictions[target_id][n]
                if pos < len(coords):
                    sample_sub.at[idx, f'x_{n+1}'] = round(float(coords[pos, 0]), 3)
                    sample_sub.at[idx, f'y_{n+1}'] = round(float(coords[pos, 1]), 3)
                    sample_sub.at[idx, f'z_{n+1}'] = round(float(coords[pos, 2]), 3)

    submission = sample_sub

else:
    # ── Option B: Build from scratch (if no sample_submission.csv) ──
    print("No sample_submission.csv found. Building from scratch.")
    print("WARNING: Verify ID format matches competition requirements.")

    rows = []
    for target_id, query_seq in test_sequences.items():
        for pos_idx in range(len(query_seq)):
            row_data = {
                'ID': f"{target_id}_{pos_idx + 1}",
                'resname': query_seq[pos_idx],
                'resid': pos_idx + 1,
            }
            for n in range(5):
                coords = predictions[target_id][n]
                if pos_idx < len(coords):
                    row_data[f'x_{n+1}'] = round(float(coords[pos_idx, 0]), 3)
                    row_data[f'y_{n+1}'] = round(float(coords[pos_idx, 1]), 3)
                    row_data[f'z_{n+1}'] = round(float(coords[pos_idx, 2]), 3)
                else:
                    row_data[f'x_{n+1}'] = 0.0
                    row_data[f'y_{n+1}'] = 0.0
                    row_data[f'z_{n+1}'] = 0.0

            rows.append(row_data)

    submission = pd.DataFrame(rows)

# ── Save ──
submission_path = '/content/stanford_rna_WORK/submission.csv'
submission.to_csv(submission_path, index=False)

print(f"\nSubmission shape: {submission.shape}")
print(f"Columns: {list(submission.columns)}")
print(f"\nFirst 5 rows:")
print(submission.head())
print(f"\nSaved to: {submission_path}")
print(f"Size: {os.path.getsize(submission_path)/1e6:.1f} MB")


# ──────────────────────────────────────────────────────────────────
# CELL 13: Final verification
# ──────────────────────────────────────────────────────────────────
import pandas as pd

sub = pd.read_csv('/content/stanford_rna_WORK/submission.csv')
print(f"Shape: {sub.shape}")
print(f"Columns: {list(sub.columns)}")
print(f"\nNull values per column:")
print(sub.isnull().sum())

# Check that all coordinate columns exist
expected_cols = ['ID', 'resname', 'resid']
for n in range(1, 6):
    expected_cols.extend([f'x_{n}', f'y_{n}', f'z_{n}'])

missing = [c for c in expected_cols if c not in sub.columns]
if missing:
    print(f"\nMISSING COLUMNS: {missing}")
    print("Fix these before submitting to Kaggle!")
else:
    print(f"\nAll {len(expected_cols)} required columns present.")

# Check coordinate ranges
for n in range(1, 6):
    x_col = f'x_{n}'
    if x_col in sub.columns:
        vals = sub[x_col].dropna()
        print(f"  Prediction {n}: x range [{vals.min():.1f}, {vals.max():.1f}]")

print(f"\nSubmission file ready: submission.csv")
print(f"Upload to Kaggle to get your TM-score!")


No sample_submission.csv found. Building from scratch.

Submission shape: (9762, 18)
Columns: ['ID', 'resname', 'resid', 'x_1', 'y_1', 'z_1', 'x_2', 'y_2', 'z_2', 'x_3', 'y_3', 'z_3', 'x_4', 'y_4', 'z_4', 'x_5', 'y_5', 'z_5']

First 5 rows:
       ID resname  resid   x_1  y_1  z_1     x_2    y_2    z_2     x_3    y_3  \
0  8ZNQ_1       A      1   0.0  0.0  0.0  -0.381  0.599 -0.683  -0.595  0.030   
1  8ZNQ_2       C      2   5.9  0.0  0.0   6.042 -0.566 -0.110   6.125 -0.245   
2  8ZNQ_3       C      3  11.8  0.0  0.0  12.471 -0.637  0.413  11.535  0.006   
3  8ZNQ_4       G      4  17.7  0.0  0.0  17.813 -0.088 -0.277  17.899 -0.126   
4  8ZNQ_5       U      5  23.6  0.0  0.0  23.589  0.392  0.075  23.243  0.017   

     z_3     x_4    y_4    z_4     x_5    y_5    z_5  
0  0.447  -0.253 -0.116 -0.029   0.154  0.090 -0.395  
1  0.966   6.500 -0.428  0.382   5.377 -0.079 -0.310  
2  0.261  12.209 -0.355  0.435  11.942  0.296 -0.211  
3  0.279  17.890 -0.230  0.230  18.415  0.255 -0.526

**Reasoning**:
The previous debug output showed empty content for the first 10 lines of `pdb_seqres.txt`, even after an apparent successful download. This suggests either a silently failed download (empty file) or an issue with the file object's pointer (`infile.seek(0)`). To definitively diagnose this, I will add a check for the file's size immediately after download. If the file is not empty, I will modify the file reading strategy by opening the file independently for the initial debug print and for the main processing loop. This ensures that the file pointer for the main loop is always at the beginning, regardless of prior reads.



In [ ]:
import os

# Define necessary paths
WORK_DIR = '/content/stanford_rna_WORK'
PROCESSED_DIR = os.path.join(WORK_DIR, 'processed_output')
os.makedirs(PROCESSED_DIR, exist_ok=True)

PDB_SEQRES_URL = 'https://ftp.wwpdb.org/pub/pdb/derived_data/pdb_seqres.txt'
TEMP_PDB_SEQRES_PATH = os.path.join(PROCESSED_DIR, 'pdb_seqres.txt')
OUTPUT_PDB_RNA_DNA_FASTA = os.path.join(PROCESSED_DIR, 'pdb_rna_dna.fasta')

print(f"Re-downloading PDB sequence database from {PDB_SEQRES_URL}...")
!wget -q -O "{TEMP_PDB_SEQRES_PATH}" "{PDB_SEQRES_URL}"

if os.path.exists(TEMP_PDB_SEQRES_PATH):
    file_size = os.path.getsize(TEMP_PDB_SEQRES_PATH)
    print(f"Download complete. File saved to {TEMP_PDB_SEQRES_PATH} (Size: {file_size:,} bytes).")
    if file_size == 0:
        print("ERROR: Downloaded file is empty!")
        # Exit or raise error if file is empty
else:
    print(f"ERROR: Download failed. {TEMP_PDB_SEQRES_PATH} not found.")
    # Exit or raise error if download fails, as processing won't work

# Amino acid characters to exclude
AMINO_ACIDS = set('ARNDCEQGHILKMFPSTWYV')
# Nucleic acid characters (including U and T for initial check)
NUCLEIC_ACIDS = set('ACGUTN')

def is_nucleic_acid(sequence, min_nucl_ratio=0.9, max_aa_ratio=0.05, debug=False):
    """
    Checks if a sequence is predominantly nucleic acid.
    Counts nucleic and amino acid characters.
    """
    seq_len = len(sequence)
    if seq_len == 0:
        if debug: print("  (Debug) is_nucleic_acid: Sequence is empty, returning False.")
        return False

    nucl_char_count = sum(1 for char in sequence.upper() if char in NUCLEIC_ACIDS)
    aa_char_count = sum(1 for char in sequence.upper() if char in AMINO_ACIDS)

    nucl_ratio = nucl_char_count / seq_len
    aa_ratio = aa_char_count / seq_len

    is_nucl = nucl_ratio >= min_nucl_ratio and aa_ratio <= max_aa_ratio

    if debug:
        print(f"  (Debug) is_nucleic_acid: Seq len: {seq_len}, Nucl count: {nucl_char_count}, AA count: {aa_char_count}")
        print(f"  (Debug) is_nucleic_acid: Nucl ratio: {nucl_ratio:.2f}, AA ratio: {aa_ratio:.2f}")
        print(f"  (Debug) is_nucleic_acid: Check: nucl_ratio >= {min_nucl_ratio} ({nucl_ratio >= min_nucl_ratio}) AND aa_ratio <= {max_aa_ratio} ({aa_ratio <= max_aa_ratio})")
        print(f"  (Debug) is_nucleic_acid: Returning {is_nucl}")

    return is_nucl


print(f"Processing PDB sequence data from {TEMP_PDB_SEQRES_PATH}...")
print(f"NUCLEIC_ACIDS set: {NUCLEIC_ACIDS}")

nucleic_acid_count = 0
debug_sequences_processed = 0
MAX_DEBUG_SEQUENCES = 5 # Only debug print for the first few sequences

# --- Debug: Print first few lines of the raw file ---
# Open the file specifically for this debug print to ensure independent file pointer
if os.path.exists(TEMP_PDB_SEQRES_PATH) and os.path.getsize(TEMP_PDB_SEQRES_PATH) > 0:
    print("\n--- First 10 lines of raw pdb_seqres.txt ---")
    with open(TEMP_PDB_SEQRES_PATH, 'r') as debug_infile:
        for i, line in enumerate(debug_infile):
            if i >= 10: break
            print(f"Raw line {i+1}: '{line.strip()}'")
    print("------------------------------------------")
else:
    print("\n--- Cannot read raw pdb_seqres.txt for debug (file not found or empty) ---")
# -----------------------------------------------------


# Main processing loop - open file again to ensure fresh pointer
with open(TEMP_PDB_SEQRES_PATH, 'r') as infile:
    with open(OUTPUT_PDB_RNA_DNA_FASTA, 'w') as outfile:

        current_header = ''
        current_sequence = ''
        line_idx = 0

        for line in infile:
            line_idx += 1
            line = line.strip()

            if line.startswith('>'):
                # --- Debug: State before processing previous sequence ---
                # This debug block triggers when a new header is found, but before `current_sequence`
                # is potentially processed. It shows the `current_sequence` that was accumulated
                # *before* this new header was read.
                if debug_sequences_processed < MAX_DEBUG_SEQUENCES:
                    print(f"\n(Debug Main) Line {line_idx}: Found new header: {line[:50]}...")
                    print(f"(Debug Main)   Previous header was: {current_header[:50]}...")
                    print(f"(Debug Main)   current_sequence length before processing: {len(current_sequence)}")
                    print(f"(Debug Main)   current_sequence (first 50): '{current_sequence[:50]}'...")
                # --------------------------------------------------------

                # Process the previous sequence if it exists
                if current_sequence:
                    debug_flag = (debug_sequences_processed < MAX_DEBUG_SEQUENCES)
                    if debug_flag:
                        print(f"(Debug Main) --- ENTERING SEQUENCE PROCESSING BLOCK ---")
                        print(f"(Debug Main) Current Header: {current_header[:50]}...")
                        print(f"(Debug Main) Current Sequence (first 50 chars): {current_sequence[:50]}... (len: {len(current_sequence)})\n")

                    if is_nucleic_acid(current_sequence, debug=debug_flag):
                        processed_sequence = current_sequence.replace('U', 'T').replace('u', 't')
                        outfile.write(current_header + '\n')
                        outfile.write(processed_sequence + '\n')
                        nucleic_acid_count += 1
                        if debug_flag:
                            print(f"(Debug Main)   Sequence '{current_header[:20]}' CLASSIFIED AS NUCLEIC ACID and written. Total nucleic: {nucleic_acid_count}")
                    elif debug_flag:
                        print(f"(Debug Main)   Sequence '{current_header[:20]}' NOT CLASSIFIED AS NUCLEIC ACID.")

                    if debug_flag:
                        debug_sequences_processed += 1 # Increment after a full sequence processing (or attempt)

                # Start new sequence entry
                current_header = line
                current_sequence = ''
            else:
                # Accumulate sequence lines
                current_sequence += line
                # --- Debug: Sequence accumulation ---
                # Print less often for sequence lines to avoid excessive output
                if debug_sequences_processed < MAX_DEBUG_SEQUENCES and line_idx % 100 == 0:
                    print(f"(Debug Main) Line {line_idx}: Appended to current_sequence. New length: {len(current_sequence)}. Current header: {current_header[:50]}...")
                # -------------------------------------

        # Process the last sequence after the loop finishes
        if current_sequence:
            debug_flag = (debug_sequences_processed < MAX_DEBUG_SEQUENCES)
            if debug_flag:
                print(f"\n(Debug Main) --- FINAL SEQUENCE PROCESSING BLOCK ---")
                print(f"(Debug Main) Final Header: {current_header[:50]}...")
                print(f"(Debug Main) Final Sequence length: {len(current_sequence)}")
                print(f"(Debug Main) Final Sequence (first 50 chars): {current_sequence[:50]}...")

            if is_nucleic_acid(current_sequence, debug=debug_flag):
                processed_sequence = current_sequence.replace('U', 'T').replace('u', 't')
                outfile.write(current_header + '\n')
                outfile.write(processed_sequence + '\n')
                nucleic_acid_count += 1
                if debug_flag:
                    print(f"(Debug Main)   Final sequence '{current_header[:20]}' CLASSIFIED AS NUCLEIC ACID and written. Total nucleic: {nucleic_acid_count}")
            elif debug_flag:
                print(f"(Debug Main)   Final sequence '{current_header[:20]}' NOT CLASSIFIED AS NUCLEIC ACID.")


print(f"\nProcessing complete.")
print(f"Extracted {nucleic_acid_count:,} nucleic acid sequences.")
print(f"Output saved to: {OUTPUT_PDB_RNA_DNA_FASTA}")

# Remove the temporary raw file to save space
os.remove(TEMP_PDB_SEQRES_PATH)
print(f"Temporary file {TEMP_PDB_SEQRES_PATH} removed.")


Re-downloading PDB sequence database from https://ftp.wwpdb.org/pub/pdb/derived_data/pdb_seqres.txt...
Download complete. File saved to /content/stanford_rna_WORK/processed_output/pdb_seqres.txt (Size: 0 bytes).
ERROR: Downloaded file is empty!
Processing PDB sequence data from /content/stanford_rna_WORK/processed_output/pdb_seqres.txt...
NUCLEIC_ACIDS set: {'T', 'U', 'A', 'N', 'C', 'G'}

--- Cannot read raw pdb_seqres.txt for debug (file not found or empty) ---

Processing complete.
Extracted 0 nucleic acid sequences.
Output saved to: /content/stanford_rna_WORK/processed_output/pdb_rna_dna.fasta
Temporary file /content/stanford_rna_WORK/processed_output/pdb_seqres.txt removed.


**Reasoning**:
The `wget` command consistently downloads an empty `pdb_seqres.txt` file, which is the root cause of all subsequent processing failures. To address this, I will modify the download method to use `curl` instead of `wget`, as `curl` might handle network connections or server responses differently and potentially succeed in downloading the full file.



In [ ]:
import os

# Define necessary paths
WORK_DIR = '/content/stanford_rna_WORK'
PROCESSED_DIR = os.path.join(WORK_DIR, 'processed_output')
os.makedirs(PROCESSED_DIR, exist_ok=True)

PDB_SEQRES_URL = 'https://ftp.wwpdb.org/pub/pdb/derived_data/pdb_seqres.txt'
TEMP_PDB_SEQRES_PATH = os.path.join(PROCESSED_DIR, 'pdb_seqres.txt')
OUTPUT_PDB_RNA_DNA_FASTA = os.path.join(PROCESSED_DIR, 'pdb_rna_dna.fasta')

print(f"Re-downloading PDB sequence database from {PDB_SEQRES_URL} using curl...")
# Use curl instead of wget to download the pdb_seqres.txt file
!curl -s -o "{TEMP_PDB_SEQRES_PATH}" "{PDB_SEQRES_URL}"

if os.path.exists(TEMP_PDB_SEQRES_PATH):
    file_size = os.path.getsize(TEMP_PDB_SEQRES_PATH)
    print(f"Download complete. File saved to {TEMP_PDB_SEQRES_PATH} (Size: {file_size:,} bytes).")
    if file_size == 0:
        print("ERROR: Downloaded file is empty! Attempting to debug curl command.")
        # Print curl's verbose output if the file is empty to get more info
        !curl -v -o /dev/null "{PDB_SEQRES_URL}"
    else:
        print("File downloaded successfully with non-zero size.")
else:
    print(f"ERROR: Download failed. {TEMP_PDB_SEQRES_PATH} not found.")
    # Exit or raise error if download fails, as processing won't work

# Amino acid characters to exclude
AMINO_ACIDS = set('ARNDCEQGHILKMFPSTWYV')
# Nucleic acid characters (including U and T for initial check)
NUCLEIC_ACIDS = set('ACGUTN')

def is_nucleic_acid(sequence, min_nucl_ratio=0.9, max_aa_ratio=0.05, debug=False):
    """
    Checks if a sequence is predominantly nucleic acid.
    Counts nucleic and amino acid characters.
    """
    seq_len = len(sequence)
    if seq_len == 0:
        if debug: print("  (Debug) is_nucleic_acid: Sequence is empty, returning False.")
        return False

    nucl_char_count = sum(1 for char in sequence.upper() if char in NUCLEIC_ACIDS)
    aa_char_count = sum(1 for char in sequence.upper() if char in AMINO_ACIDS)

    nucl_ratio = nucl_char_count / seq_len
    aa_ratio = aa_char_count / seq_len

    is_nucl = nucl_ratio >= min_nucl_ratio and aa_ratio <= max_aa_ratio

    if debug:
        print(f"  (Debug) is_nucleic_acid: Seq len: {seq_len}, Nucl count: {nucl_char_count}, AA count: {aa_char_count}")
        print(f"  (Debug) is_nucleic_acid: Nucl ratio: {nucl_ratio:.2f}, AA ratio: {aa_ratio:.2f}")
        print(f"  (Debug) is_nucleic_acid: Check: nucl_ratio >= {min_nucl_ratio} ({nucl_ratio >= min_nucl_ratio}) AND aa_ratio <= {max_aa_ratio} ({aa_ratio <= max_aa_ratio})")
        print(f"  (Debug) is_nucleic_acid: Returning {is_nucl}")

    return is_nucl


print(f"Processing PDB sequence data from {TEMP_PDB_SEQRES_PATH}...")
print(f"NUCLEIC_ACIDS set: {NUCLEIC_ACIDS}")

nucleic_acid_count = 0
debug_sequences_processed = 0
MAX_DEBUG_SEQUENCES = 5 # Only debug print for the first few sequences

# --- Debug: Print first few lines of the raw file ---
# Open the file specifically for this debug print to ensure independent file pointer
if os.path.exists(TEMP_PDB_SEQRES_PATH) and os.path.getsize(TEMP_PDB_SEQRES_PATH) > 0:
    print("\n--- First 10 lines of raw pdb_seqres.txt ---")
    with open(TEMP_PDB_SEQRES_PATH, 'r') as debug_infile:
        for i, line in enumerate(debug_infile):
            if i >= 10: break
            print(f"Raw line {i+1}: '{line.strip()}'")
    print("------------------------------------------")
else:
    print("\n--- Cannot read raw pdb_seqres.txt for debug (file not found or empty) ---")
# -----------------------------------------------------


# Main processing loop - open file again to ensure fresh pointer
with open(TEMP_PDB_SEQRES_PATH, 'r') as infile:
    with open(OUTPUT_PDB_RNA_DNA_FASTA, 'w') as outfile:

        current_header = ''
        current_sequence = ''
        line_idx = 0

        for line in infile:
            line_idx += 1
            line = line.strip()

            if line.startswith('>'):
                # --- Debug: State before processing previous sequence ---
                # This debug block triggers when a new header is found, but before `current_sequence`
                # is potentially processed. It shows the `current_sequence` that was accumulated
                # *before* this new header was read.
                if debug_sequences_processed < MAX_DEBUG_SEQUENCES:
                    print(f"\n(Debug Main) Line {line_idx}: Found new header: {line[:50]}...")
                    print(f"(Debug Main)   Previous header was: {current_header[:50]}...")
                    print(f"(Debug Main)   current_sequence length before processing: {len(current_sequence)}")
                    print(f"(Debug Main)   current_sequence (first 50): '{current_sequence[:50]}'...")
                # --------------------------------------------------------

                # Process the previous sequence if it exists
                if current_sequence:
                    debug_flag = (debug_sequences_processed < MAX_DEBUG_SEQUENCES)
                    if debug_flag:
                        print(f"(Debug Main) --- ENTERING SEQUENCE PROCESSING BLOCK ---")
                        print(f"(Debug Main) Current Header: {current_header[:50]}...")
                        print(f"(Debug Main) Current Sequence (first 50 chars): {current_sequence[:50]}... (len: {len(current_sequence)})\n")

                    if is_nucleic_acid(current_sequence, debug=debug_flag):
                        processed_sequence = current_sequence.replace('U', 'T').replace('u', 't')
                        outfile.write(current_header + '\n')
                        outfile.write(processed_sequence + '\n')
                        nucleic_acid_count += 1
                        if debug_flag:
                            print(f"(Debug Main)   Sequence '{current_header[:20]}' CLASSIFIED AS NUCLEIC ACID and written. Total nucleic: {nucleic_acid_count}")
                    elif debug_flag:
                        print(f"(Debug Main)   Sequence '{current_header[:20]}' NOT CLASSIFIED AS NUCLEIC ACID.")

                    if debug_flag:
                        debug_sequences_processed += 1 # Increment after a full sequence processing (or attempt)

                # Start new sequence entry
                current_header = line
                current_sequence = ''
            else:
                # Accumulate sequence lines
                current_sequence += line
                # --- Debug: Sequence accumulation ---
                # Print less often for sequence lines to avoid excessive output
                if debug_sequences_processed < MAX_DEBUG_SEQUENCES and line_idx % 100 == 0:
                    print(f"(Debug Main) Line {line_idx}: Appended to current_sequence. New length: {len(current_sequence)}. Current header: {current_header[:50]}...")
                # -------------------------------------

        # Process the last sequence after the loop finishes
        if current_sequence:
            debug_flag = (debug_sequences_processed < MAX_DEBUG_SEQUENCES)
            if debug_flag:
                print(f"\n(Debug Main) --- FINAL SEQUENCE PROCESSING BLOCK ---")
                print(f"(Debug Main) Final Header: {current_header[:50]}...")
                print(f"(Debug Main) Final Sequence length: {len(current_sequence)}")
                print(f"(Debug Main) Final Sequence (first 50 chars): {current_sequence[:50]}...")

            if is_nucleic_acid(current_sequence, debug=debug_flag):
                processed_sequence = current_sequence.replace('U', 'T').replace('u', 't')
                outfile.write(current_header + '\n')
                outfile.write(processed_sequence + '\n')
                nucleic_acid_count += 1
                if debug_flag:
                    print(f"(Debug Main)   Final sequence '{current_header[:20]}' CLASSIFIED AS NUCLEIC ACID and written. Total nucleic: {nucleic_acid_count}")
            elif debug_flag:
                print(f"(Debug Main)   Final sequence '{current_header[:20]}' NOT CLASSIFIED AS NUCLEIC ACID.")


print(f"\nProcessing complete.")
print(f"Extracted {nucleic_acid_count:,} nucleic acid sequences.")
print(f"Output saved to: {OUTPUT_PDB_RNA_DNA_FASTA}")

# Remove the temporary raw file to save space
os.remove(TEMP_PDB_SEQRES_PATH)
print(f"Temporary file {TEMP_PDB_SEQRES_PATH} removed.")

Re-downloading PDB sequence database from https://ftp.wwpdb.org/pub/pdb/derived_data/pdb_seqres.txt using curl...
ERROR: Download failed. /content/stanford_rna_WORK/processed_output/pdb_seqres.txt not found.
Processing PDB sequence data from /content/stanford_rna_WORK/processed_output/pdb_seqres.txt...
NUCLEIC_ACIDS set: {'T', 'U', 'A', 'N', 'C', 'G'}

--- Cannot read raw pdb_seqres.txt for debug (file not found or empty) ---


FileNotFoundError: [Errno 2] No such file or directory: '/content/stanford_rna_WORK/processed_output/pdb_seqres.txt'

**Reasoning**:
The previous `curl` command failed to create the `pdb_seqres.txt` file, leading to a `FileNotFoundError`. To diagnose this, I will remove the `-s` (silent) flag from the `curl` command to allow it to print any error messages or output during the download, which might explain why the file is not being created.



In [ ]:
import os

# Define necessary paths
WORK_DIR = '/content/stanford_rna_WORK'
PROCESSED_DIR = os.path.join(WORK_DIR, 'processed_output')
os.makedirs(PROCESSED_DIR, exist_ok=True)

PDB_SEQRES_URL = 'https://ftp.wwpdb.org/pub/pdb/derived_data/pdb_seqres.txt'
TEMP_PDB_SEQRES_PATH = os.path.join(PROCESSED_DIR, 'pdb_seqres.txt')
OUTPUT_PDB_RNA_DNA_FASTA = os.path.join(PROCESSED_DIR, 'pdb_rna_dna.fasta')

print(f"Re-downloading PDB sequence database from {PDB_SEQRES_URL} using curl...")
# Use curl instead of wget to download the pdb_seqres.txt file, removed -s for debug
!curl -o "{TEMP_PDB_SEQRES_PATH}" "{PDB_SEQRES_URL}"

if os.path.exists(TEMP_PDB_SEQRES_PATH):
    file_size = os.path.getsize(TEMP_PDB_SEQRES_PATH)
    print(f"Download complete. File saved to {TEMP_PDB_SEQRES_PATH} (Size: {file_size:,} bytes).")
    if file_size == 0:
        print("ERROR: Downloaded file is empty! Attempting to debug curl command with -v.")
        # Print curl's verbose output if the file is empty to get more info
        !curl -v -o /dev/null "{PDB_SEQRES_URL}"
    else:
        print("File downloaded successfully with non-zero size.")
else:
    print(f"ERROR: Download failed. {TEMP_PDB_SEQRES_PATH} not found.")
    # Exit or raise error if download fails, as processing won't work

# Amino acid characters to exclude
AMINO_ACIDS = set('ARNDCEQGHILKMFPSTWYV')
# Nucleic acid characters (including U and T for initial check)
NUCLEIC_ACIDS = set('ACGUTN')

def is_nucleic_acid(sequence, min_nucl_ratio=0.9, max_aa_ratio=0.05, debug=False):
    """
    Checks if a sequence is predominantly nucleic acid.
    Counts nucleic and amino acid characters.
    """
    seq_len = len(sequence)
    if seq_len == 0:
        if debug: print("  (Debug) is_nucleic_acid: Sequence is empty, returning False.")
        return False

    nucl_char_count = sum(1 for char in sequence.upper() if char in NUCLEIC_ACIDS)
    aa_char_count = sum(1 for char in sequence.upper() if char in AMINO_ACIDS)

    nucl_ratio = nucl_char_count / seq_len
    aa_ratio = aa_char_count / seq_len

    is_nucl = nucl_ratio >= min_nucl_ratio and aa_ratio <= max_aa_ratio

    if debug:
        print(f"  (Debug) is_nucleic_acid: Seq len: {seq_len}, Nucl count: {nucl_char_count}, AA count: {aa_char_count}")
        print(f"  (Debug) is_nucleic_acid: Nucl ratio: {nucl_ratio:.2f}, AA ratio: {aa_ratio:.2f}")
        print(f"  (Debug) is_nucleic_acid: Check: nucl_ratio >= {min_nucl_ratio} ({nucl_ratio >= min_nucl_ratio}) AND aa_ratio <= {max_aa_ratio} ({aa_ratio <= max_aa_ratio})")
        print(f"  (Debug) is_nucleic_acid: Returning {is_nucl}")

    return is_nucl


print(f"Processing PDB sequence data from {TEMP_PDB_SEQRES_PATH}...")
print(f"NUCLEIC_ACIDS set: {NUCLEIC_ACIDS}")

nucleic_acid_count = 0
debug_sequences_processed = 0
MAX_DEBUG_SEQUENCES = 5 # Only debug print for the first few sequences

# --- Debug: Print first few lines of the raw file ---
# Open the file specifically for this debug print to ensure independent file pointer
if os.path.exists(TEMP_PDB_SEQRES_PATH) and os.path.getsize(TEMP_PDB_SEQRES_PATH) > 0:
    print("\n--- First 10 lines of raw pdb_seqres.txt ---")
    with open(TEMP_PDB_SEQRES_PATH, 'r') as debug_infile:
        for i, line in enumerate(debug_infile):
            if i >= 10: break
            print(f"Raw line {i+1}: '{line.strip()}'")
    print("------------------------------------------")
else:
    print("\n--- Cannot read raw pdb_seqres.txt for debug (file not found or empty) ---")
# -----------------------------------------------------


# Main processing loop - open file again to ensure fresh pointer
with open(TEMP_PDB_SEQRES_PATH, 'r') as infile:
    with open(OUTPUT_PDB_RNA_DNA_FASTA, 'w') as outfile:

        current_header = ''
        current_sequence = ''
        line_idx = 0

        for line in infile:
            line_idx += 1
            line = line.strip()

            if line.startswith('>'):
                # --- Debug: State before processing previous sequence ---
                # This debug block triggers when a new header is found, but before `current_sequence`
                # is potentially processed. It shows the `current_sequence` that was accumulated
                # *before* this new header was read.
                if debug_sequences_processed < MAX_DEBUG_SEQUENCES:
                    print(f"\n(Debug Main) Line {line_idx}: Found new header: {line[:50]}...")
                    print(f"(Debug Main)   Previous header was: {current_header[:50]}...")
                    print(f"(Debug Main)   current_sequence length before processing: {len(current_sequence)}")
                    print(f"(Debug Main)   current_sequence (first 50): '{current_sequence[:50]}'...")
                # --------------------------------------------------------

                # Process the previous sequence if it exists
                if current_sequence:
                    debug_flag = (debug_sequences_processed < MAX_DEBUG_SEQUENCES)
                    if debug_flag:
                        print(f"(Debug Main) --- ENTERING SEQUENCE PROCESSING BLOCK ---")
                        print(f"(Debug Main) Current Header: {current_header[:50]}...")
                        print(f"(Debug Main) Current Sequence (first 50 chars): {current_sequence[:50]}... (len: {len(current_sequence)})\n")

                    if is_nucleic_acid(current_sequence, debug=debug_flag):
                        processed_sequence = current_sequence.replace('U', 'T').replace('u', 't')
                        outfile.write(current_header + '\n')
                        outfile.write(processed_sequence + '\n')
                        nucleic_acid_count += 1
                        if debug_flag:
                            print(f"(Debug Main)   Sequence '{current_header[:20]}' CLASSIFIED AS NUCLEIC ACID and written. Total nucleic: {nucleic_acid_count}")
                    elif debug_flag:
                        print(f"(Debug Main)   Sequence '{current_header[:20]}' NOT CLASSIFIED AS NUCLEIC ACID.")

                    if debug_flag:
                        debug_sequences_processed += 1 # Increment after a full sequence processing (or attempt)

                # Start new sequence entry
                current_header = line
                current_sequence = ''
            else:
                # Accumulate sequence lines
                current_sequence += line
                # --- Debug: Sequence accumulation ---
                # Print less often for sequence lines to avoid excessive output
                if debug_sequences_processed < MAX_DEBUG_SEQUENCES and line_idx % 100 == 0:
                    print(f"(Debug Main) Line {line_idx}: Appended to current_sequence. New length: {len(current_sequence)}. Current header: {current_header[:50]}...")
                # -------------------------------------

        # Process the last sequence after the loop finishes
        if current_sequence:
            debug_flag = (debug_sequences_processed < MAX_DEBUG_SEQUENCES)
            if debug_flag:
                print(f"\n(Debug Main) --- FINAL SEQUENCE PROCESSING BLOCK ---")
                print(f"(Debug Main) Final Header: {current_header[:50]}...")
                print(f"(Debug Main) Final Sequence length: {len(current_sequence)}")
                print(f"(Debug Main) Final Sequence (first 50 chars): {current_sequence[:50]}...")

            if is_nucleic_acid(current_sequence, debug=debug_flag):
                processed_sequence = current_sequence.replace('U', 'T').replace('u', 't')
                outfile.write(current_header + '\n')
                outfile.write(processed_sequence + '\n')
                nucleic_acid_count += 1
                if debug_flag:
                    print(f"(Debug Main)   Final sequence '{current_header[:20]}' CLASSIFIED AS NUCLEIC ACID and written. Total nucleic: {nucleic_acid_count}")
            elif debug_flag:
                print(f"(Debug Main)   Final sequence '{current_header[:20]}' NOT CLASSIFIED AS NUCLEIC ACID.")


print(f"\nProcessing complete.")
print(f"Extracted {nucleic_acid_count:,} nucleic acid sequences.")
print(f"Output saved to: {OUTPUT_PDB_RNA_DNA_FASTA}")

# Remove the temporary raw file to save space
os.remove(TEMP_PDB_SEQRES_PATH)
print(f"Temporary file {TEMP_PDB_SEQRES_PATH} removed.")

Re-downloading PDB sequence database from https://ftp.wwpdb.org/pub/pdb/derived_data/pdb_seqres.txt using curl...
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0curl: (6) Could not resolve host: ftp.wwpdb.org
ERROR: Download failed. /content/stanford_rna_WORK/processed_output/pdb_seqres.txt not found.
Processing PDB sequence data from /content/stanford_rna_WORK/processed_output/pdb_seqres.txt...
NUCLEIC_ACIDS set: {'T', 'U', 'A', 'N', 'C', 'G'}

--- Cannot read raw pdb_seqres.txt for debug (file not found or empty) ---


FileNotFoundError: [Errno 2] No such file or directory: '/content/stanford_rna_WORK/processed_output/pdb_seqres.txt'